# Projeto - Simulando a Opinião Pública
## Tema: Democracia
## Modelo escolhido: Gemma 3, 1B de parâmetros

**Identificação do Grupo**

* Bruna Aguiar Muchiuti - 10418358
* Gabriel Ken Kazama Geronazzo - 10418247
* Enzo Benedetto Proença - 10418579
* Lucas Pires de Camargo Sarai - 10418013 
* Jessica dos Santos Santana Bispo - 10410798
* Otávio Augusto Freire de Andrade Bruzadin - 10409053

## Primeiros passos
### Antes de executar as células de código, siga os passos a seguir
**Baixar e preparar Ollama:**
1. Escolha o comando apropriado para o seu SO (MacOS, Linux, Windows) no site do [Ollama](https://ollama.com/download)
> Para Linux/MacOS, pode ser necessário executar o comando abaixo
> ```bash
>  sudo apt-get install zstd
> ```
2.  Execute o servidor do Ollama localmente, o que deve abrir a porta `localhost:11434`
    ```bash
    ollama serve
    ```
> OBS: Se retornar que a porta já está em uso, é porque após o download, já foi executado automaticamente o comando anterior. Para confirmar, acesse `http://localhost:11434` no navegador e veja se retorna *Ollama is running* 
3. Execute o comando abaixo para baixar e executar o modelo:
    ```bash
    ollama run gemma3:1b "preload" > /dev/null
    ```
> OBS: Se não funcionar, execute `ollama pull gemma3:1b antes`

**Preparar ambiente virtual do Python:**<br><br>
Para baixar as dependências apenas para esse projeto (e não de forma fixa na sua máquina) execute essas instruções:
```bash
python3 -m venv .venv;
source .venv/bin/activate;
```

### Baixando bibliotecas necessárias

* **Pandas:** Para trabalhar com a base de dados
* **PyReadStat:** Para trabalhar com a base de dados, permitindo ler arquivo .sav e complementar o pandas
* **Ollama:** API do LLM
* **Pydantic + Typing**: Validação de formato de saída JSON

In [1]:
# Instalar a lib de Python para Ollama
!pip install ollama

In [2]:
# Instalar pndas
!pip install pandas

In [3]:
# Instalar a lib para ler arquivos .sav
!pip install pyreadstat

In [4]:
!pip install pydantic

In [5]:
!pip install seaborn

In [6]:
!pip install matplotlib

In [7]:
!pip install scikit-learn

### Desenvolvimento da conversa com o modelo

#### Parte 1) Selecionar variáveis mais relevantes
O primeiro passo é pedir ao LLM reconhecer quais são as variáveis mais relevantes para ele na hora de simular o participante.

Para isso, para cada pergunta, será enviado um prompt indicando essa tarefa, as variáveis com valores possíveis, a pergunta e um JSON Schema de resposta esperada.

##### Entrada
**Prompt de Sistema**
> "*Você é um assistente de raciocínio treinado para entender como características pessoais moldam crenças. Sua tarefa é analisar características e selecionar as mais preditivas para o tópico de Democracia.*
>
> *Você receberá uma pergunta, suas possíveis respostas e se é de múltipla escolha ou escolha única, e deverá retornar uma lista de características que julgar que são as mais relevantes para responder a pergunta.*
> *Características possíveis: <lista de características, ignorando as outras perguntas>*

**Prompt de Entrada**
> *Pergunta: ...,Respostas Possíveis: [...], Múltipla Escolhda: Sim | Não*


##### Saída
```json
  {
    "question" : {
      "type": "string"
    },
    "caracteristicas_selecionadas": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "minItems": 1,
        "uniqueItems": true,
        "default": [
          "Nenhuma é relevante"
        ]
      },
  }
```


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_spss("04832.SAV")

df.head()

,SEXO,IDADE,FX_ID,ESCOLARIDADE,P1A,P1B,P1C,P2_1,P2_2,P2_3,...,RACA,RELIGIAO,REND1,REND2,REGIAO,COND,PORTE,ID_Ipec,DATA_ENTREVISTA,TIPO_COLETA
0,FEM,29.0,25 A 34,Superior completo,Sim,Sim,Sim,Reduzir as desigualdades sociais,Reduzir a violência,Melhorar a qualidade da saúde,...,Branca,Outras Evangélicas específicas,MAIS DE 2 A 5,MAIS DE 10 A 20,NORDESTE,CAPITAL,MAIS DE 500.000,189519187.0,2023-09-01,Face a face
1,MAS,77.0,65 E MAIS,1ª série (ou 2º ano),Não,Não,Não,Reduzir a violência,Melhorar a qualidade da saúde,Melhorar a qualidade da educação,...,Parda,Católica Apostólica Romana,MAIS DE 2 A 5,MAIS DE 2 A 5,CENTRO OESTE,CAPITAL,MAIS DE 500.000,189525808.0,2023-09-01,Face a face
2,FEM,46.0,45 A 54,3ª série,Não,Não,Não,Reduzir as desigualdades sociais,Melhorar a qualidade da saúde,Combater as mudanças climáticas/desmatamento,...,Parda,Outras Evangélicas específicas,ATÉ 1,ATÉ 1,SUL,PERIFERIA,DE 100.001 A 500.000,189526277.0,2023-09-01,Face a face
3,MAS,72.0,65 E MAIS,3ª série (ou 4º ano),Não,Não,Não,Reduzir as desigualdades sociais,"Combater o preconceito (racismo, homofobia, di...",Incentivar a geração de empregos,...,Parda,Católica Apostólica Romana,ATÉ 1,ATÉ 1,SUDESTE,INTERIOR,DE 20.001 A 50.000,189532152.0,2023-09-01,Face a face
4,FEM,53.0,45 A 54,Superior completo,Não,Não,Não,Incentivar a geração de empregos,Melhorar a qualidade da saúde,Melhorar a qualidade da educação,...,Parda,Evangélica - Não sabe especificar,NÃO TEM RENDIMENTO PESSOAL,ATÉ 1,SUDESTE,INTERIOR,DE 20.001 A 50.000,189532154.0,2023-09-01,Face a face


In [9]:
#Investigando as características dos participantes e seus valores únicos
print("Colunas possíveis:", list(df.columns))

features_ignore: list[str] = ["ID_Ipec", "DATA_ENTREVISTA", "TIPO_COLETA","IDADE","REND1","REND2"]

#Excluindo as perguntas e características irrelevantes
feature_renda: pd.Series = pd.concat([df['REND1'].astype(str), df['REND2'].astype(str)], ignore_index=True).rename('RENDA')
participant_features: list[str] = list(df.columns[~df.columns.str.contains(r"P\d\D*")])
participant_features = [feature for feature in participant_features if feature not in features_ignore]
participant_features_values: dict[str, list[str]] = {feature: list(df[feature].unique()) for feature in participant_features}
participant_features.append(str(feature_renda.name))
participant_features_values['RENDA'] = list(feature_renda.unique())
print("Características dos participantes:", participant_features)
print("Valores únicos para cada característica:")
for feature, values in participant_features_values.items():
    print(f"  {feature}: {values}")

Colunas possíveis: ['SEXO', 'IDADE', 'FX_ID', 'ESCOLARIDADE', 'P1A', 'P1B', 'P1C', 'P2_1', 'P2_2', 'P2_3', 'P3_1', 'P3_2', 'P3_3', 'P3_4', 'P3_5', 'P3_6', 'P4', 'RACA', 'RELIGIAO', 'REND1', 'REND2', 'REGIAO', 'COND', 'PORTE', 'ID_Ipec', 'DATA_ENTREVISTA', 'TIPO_COLETA']
Características dos participantes: ['SEXO', 'FX_ID', 'ESCOLARIDADE', 'RACA', 'RELIGIAO', 'REGIAO', 'COND', 'PORTE', 'RENDA']
Valores únicos para cada característica:
  SEXO: ['FEM', 'MAS']
  FX_ID: ['25 A 34', '65 E MAIS', '45 A 54', '55 A 64', '35 A 44', '18 A 24', '16 E 17']
  ESCOLARIDADE: ['Superior completo', '1ª série (ou 2º ano)', '3ª série', '3ª série (ou 4º ano)', '4ª série (ou 5º ano)', '5ª série (ou 6º ano)', 'Superior incompleto', '2ª série (ou 3º ano)', '2ª série', 'Sabe ler/ escrever, mas não cursou escola', '8ª série (ou 9º ano)', 'Analfabeto', '1ª série', '7ª série (ou 8º ano)', '6ª série (ou 7º ano)', 'Pré-escola (ou 1º ano)']
  RACA: ['Branca', 'Parda', 'Preta', 'Indígena', 'Amarela']
  RELIGIAO: ['Out

In [10]:
questions = [
        {
            "code": "P1A",
            "question": "Você se lembra em quem votou para Deputado(a) Estadual nas eleições gerais de 2022?",
            "possible_answers": {
                "Sim": 1,
                "Não": 2,
            },
            "multiple_choice": False
        },
        {
            "code": "P1B",
            "question": "Você se lembra em quem votou para Deputado(a) Federal nas eleições gerais de 2022?",
            "possible_answers": {
                "Sim": 1,
                "Não": 2,
            },
            "multiple_choice": False
        },
        {
            "code": "P1C",
            "question": "Você se lembra em quem votou para Senador(a) Federal nas eleições gerais de 2022?",
            "possible_answers": {
                "Sim": 1,
                "Não": 2,
            },
            "multiple_choice": False
        },
        {
            "code": "P3",
            "question": " Algumas pessoas dizem que a divulgação de fake news, ou seja, de notícias ou conteúdos falsos podem prejudicar a democracia. "
                        "Pensando nisso, quais dessas opções você acredita que poderiam contribuir no combate à divulgação de fake news? ",
            "possible_answers": {
                "Ampliar a regulamentação, as regras a serem cumpridas pelas plataformas digitais (Facebook, Youtube, WhatsApp, etc.)": 1,
                "Responsabilizar e punir as empresas de tecnologia/comunicação que não removerem postagens com conteúdos falsos": 2,
                "Ampliar a regulamentação para usuários que divulgam fake news, criadas por eles próprios ou por terceiros": 3,
                "Responsabilizar e punir os usuários que divulgam ou compartilham postagens com notícias ou conteúdos falsos": 4,
                "Ampliar a regulamentação para políticos que divulgam fake news, criadas por eles próprios ou por terceiros": 5,
                "Responsabilizar, punir ou caçar políticos que divulgam ou compartilham postagens com notícias ou conteúdos falsos": 6,
            },
            "multiple_choice": True,
            "max_choices": 3
        }
]

In [11]:
from typing import Set, Literal
from pydantic import BaseModel, Field

class ExpectedAnswerVariables(BaseModel):
    features_selected: Set[Literal["SEXO", "FX_ID", "ESCOLARIDADE", "RACA", "RENDA", "REGIAO", "RELIGIAO"]] = Field(
        min_length=1,
        description="Lista de características selecionadas pelo modelo, que devem ser as mais relevantes para responder a pergunta.",

    )

In [12]:
import ollama
import json


#Definindo Host do Ollama
ollama_host = 'http://localhost:11434'
ollama.Client(host=ollama_host)

participant_features_prompt = """
SEXO: Masculino, Feminino
FX_ID: Faixa de idade do participante, em décadas 
ESCOLARIDADE: Grau de escolaridade do participante
RACA: Raça do participante
RENDA: Faixa de renda mensal do participante, em milhares de reais
REGIAO: Região geográfica do país onde o participante reside
RELIGIAO: Religião do participante
COND: Se o participante vive na Capital, Interior ou Periferia
PORTE: Patrimônio financeiro do participante, em milhares de reais
"""

# Usando prompt de sistema
system_prompt = f"""
Você é um assistente de raciocínio treinado para entender como características pessoais moldam crenças. Sua tarefa é analisar características e selecionar as mais preditivas para o tópico de Democracia.*
Você receberá uma pergunta, suas possíveis respostas e se é de múltipla escolha ou escolha única, e deverá retornar uma lista de características que julgar que são as mais relevantes para responder a pergunta.*
Características possíveis: \n{participant_features_prompt}
IMPORTANTE: 
- RESPONDA APENAS COM O NOME DA CARACTERÍSTICA SELECIONADA, o qual está em LETRAS CAPITAIS na lista acima.
- Você deve APENAS RETORNAR A LISTA DE CARACTERÍSTICAS SELECIONADAS, sem explicações ou justificativas. 
- NÃO adicione nenhuma caracterísitca que não esteja na lista.
- NÃO adicione acentuação às características, mesmo que estejam acentuadas na lista. Por exemplo, escreva "RACA" ao invés de "RAÇA", "REGIAO" ao invés de "REGIÃO", e "RELIGIAO" ao invés de "RELIGIÃO".
"""

features_selected_for_question: dict[str, list[str]] = {}

# Chama da API do Ollama
for question in questions:
    user_question: str = f"""
    Pergunta: {question['question']}
    Possíveis respostas: {', '.join(question['possible_answers'].keys())}
    Múltipla escolha: {'Sim' if question['multiple_choice'] else 'Não'}
    """
    try:
        response = ollama.chat(
            model='gemma3:1b',
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_question},
            ],
            format=ExpectedAnswerVariables.model_json_schema(),
            options={"temperature": 0.3}
        )
        print("#" * 50)
        print(f"Pergunta Lida: {question['question']}")
        print("Resposta do Gemma:\n")
        features_selected_for_question[question['code']] = [feature.strip().upper() for feature in set(json.loads(response['message']['content'])['features_selected'])]
        print(f"Características selecionadas: {features_selected_for_question[question['code']]}")
        print("#" * 50)
    except Exception as e:
        print(f"Erro ao obter resposta: {e}")

Erro ao obter resposta: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Erro ao obter resposta: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Erro ao obter resposta: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Erro ao obter resposta: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


#### Parte 2) Simular participante
O segundo passo é pedir ao LLM responder às perguntas.

Para isso, para cada pergunta, será enviado um prompt indicando essa tarefa, o perfil do respondente e a pergunta

##### Entrada
**Prompt de Sistema**
> "*Você é um participante de um questionário sobre Democracia. Sua tarefa é responder a pergunta fornecida, considerando o perfil passdo do participante que você está
> representando.*
> *IMPORTANTE: Responda apenas com a alternativa escolhida (ou com as alternativas, se for de múltipla escolha). NUNCA responda com uma alternativa que não esteja na lista

**Prompt de Entrada**
> *Perfil: [...]*<br>
>  *Pergunta: ...,Respostas Possíveis: [...], Múltipla Escolhda: Sim | Não*

##### Saída
**Escolha única**
```json
  {
    "answer": {
        "type": "integer"
    }
  }
```

**Múltipla escolha**
```json
  {
    "answers": {
        "type": "array",
        "items": {
          "type": "integer"
        },
        "minItems": 1,
        "maxItems": 3,
        "uniqueItems": true,
      },
  }
```

##### Observações

* Para a geração em massa e avaliação do desempenho, as respostas são geradas em lotes sequencias de 50,100 e 200 respondentes, onde cada resposta é apenas o código
da alternativa.
* A questão de múltipla escolha define, em seu enunciado, o limite de escolha de 3 alternativas. No entanto, alguns registros possuem 6. Para eles, foram considerados apenas os 3 primeiros.


In [13]:
class ExpectedSingleAnswer(BaseModel):
    answer: int = Field(
        description="Resposta selecionada pelo modelo, que deve ser a mais relevante para responder a pergunta. RETORNE APENAS O CÓDIGO DA ALTERNATIVA, SEM O TEXTO DESCRITIVO"
    )

In [14]:
class ExpectedMultipleAnswer(BaseModel):
    answers: Set[int] = Field(
        min_length=1,
        max_length=3,
        description="Lista de respostas selecionadas pelo modelo, que devem ser as mais relevantes para responder a pergunta. As respostas devem ser escolhidas entre as opções de resposta fornecidas para cada pergunta. RETORNE APENAS O CÓDIGO DAS ALTERNATIVAS, SEM O TEXTO DESCRITIVO"
    )

In [15]:
import re

def sort_renda(renda_unique_values: list[str]) -> list[str]:
    """
    Ordena as faixas de renda de forma lógica, considerando os termos "ATÉ", "MAIS DE" e "NÃO...".
   
    Args:
        renda_unique_values: Lista de faixas de renda em formato string, como "ATÉ
    
    Returns:
        Lista de faixas de renda ordenada logicamente.
    """
    def sort_key(s):

        #Extraindo número da palavra
        nums = list(map(int, re.findall(r'\d+', s)))
        
        if "ATÉ" in s:
            return (0, nums[0])  # menor prioridade, representa intervalo iniciado em 0
        
        elif "MAIS DE" in s:
            if len(nums) == 1:
                return (1, nums[0]) # ex: MAIS DE 20
            else:
                return (1, nums[0], nums[1])  # ex: MAIS DE 2 A 5
        
        else:
            return (-1, float('inf'))  # casos tipo "NÃO..." são os menores, equivalem a 0
    return sorted(renda_unique_values, key=sort_key)

def generate_profile(line_idx: int, participant_features: list[str], possible_answers: tuple[str,dict], df: pd.DataFrame) -> tuple[str, set[int]]:
    """
    Procura um perfil de participante baseado nas features e valores selecionadas.
    
    Args:
        line_idx: Índice da linha no DataFrame
        participant_features: Lista de características a incluir no perfil
        possible_answers: Tupla contendo o código da pergunta e o dicionário de possíveis respostas e suas codificações numéricas
        df: DataFrame com os dados dos participantes
    
    Returns:
        String formatada com o perfil do participante além de suas respostas para a pergunta específica.
    """
    # Pegar o registro correspondente ao índice
    random_record = df.iloc[line_idx]

    #Preencher com respostas do participante para a pergunta específica
    answers: set[int] = set()
    if possible_answers[0] in ('P1A', 'P1B', 'P1C'):
        answer = possible_answers[1].get(random_record[possible_answers[0]], -1)
        if answer != -1:
            answers.add(answer) # Acessa a resposta do participante para a pergunta de escolha única
    else:       
        for i in range(1, 4):
            if not pd.isna(random_record[f'P3_{i}']):
                answer = possible_answers[1].get(random_record[f'P3_{i}'], -1)
                if answer != -1:
                    answers.add(answer) # Acessa as respostas do participante para a pergunta de múltipla escolha

    # Extrair apenas as características relevantes
    profile = {}
    for feature in participant_features:
        if feature != 'RENDA':
            profile[feature] = random_record[feature]
        else:
            renda = sort_renda([str(random_record['REND1']), str(random_record['REND2'])])
            profile[feature] = f"{renda[0]} a {renda[1]} mil reais"
    
    # Formatando o perfil para exibição
    profile_str = "\n".join([f"{car}: {valor}" for car, valor in profile.items()])
    
    return profile_str, answers

In [16]:
class QuestionResults:
    """Classe para armazenar resultados de uma pergunta para posterior construção de matrizes de confusão"""
    def __init__(self, question_code: str, is_multiple_choice: bool, possible_answers: dict):
        self.code = question_code
        self.is_multiple_choice = is_multiple_choice
        self.possible_answers = possible_answers
        self.real_answers_list = []
        self.predicted_answers_list = []
        self.real_pred_pairs = []
        self.alternative_scores: dict[int, dict[str, int]] = {}
        for alt_code in possible_answers.values():
            self.alternative_scores[alt_code] = {'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0}
        self.score = 0
        self.total_processed = 0


In [17]:
def generate_answers_batch(questions: list[dict], participant_features: list[str], df: pd.DataFrame, batch_size: int) -> dict[str, QuestionResults]:
    """
    Gera respostas simuladas e coleta dados para posteriores matrizes de confusão.
    Não envia requisições repetidas.
    
    Returns:
        Dicionário com código da pergunta -> QuestionResults
    """
    system_prompt = """
    Você é um participante de um questionário sobre Democracia. Sua tarefa é responder à pergunta fornecida, considerando o perfil do participante que você está representando.
    IMPORTANTE: Responda apenas com o código da alternativa escolhida (ou com os códigos das alternativas, se for de múltipla escolha). NUNCA responda com uma alternativa que não esteja na lista
    """
    
    results: dict[str, QuestionResults] = {}
    
    for question in questions:
        answer_key: str = 'answers' if question['multiple_choice'] else 'answer'
        score_weight: int = 3 if question['multiple_choice'] else 1
        
        q_result = QuestionResults(question['code'], question['multiple_choice'], question['possible_answers'])

        for idx in range(batch_size):
            profile: tuple[str, set[int]] = generate_profile(idx, participant_features, (question['code'], question['possible_answers']), df)
            real_answers = profile[1]
            
            # Pular se não tem resposta real
            if len(real_answers) == 0:
                continue
            
            user_question: str = f"""
            Perfil do participante: {profile[0]}
            Pergunta: {question['question']}
            Possíveis respostas: {', '.join(str(v) + ': ' + k for k, v in question['possible_answers'].items())}
            Múltipla escolha: {'Sim' if question['multiple_choice'] else 'Não'}
            Máximo de escolhas permitidas: {question['max_choices'] if question['multiple_choice'] else 'N/A'}
            """
            try:
                response = ollama.chat(
                    model='gemma3:1b',
                    messages=[
                        {'role': 'system', 'content': system_prompt},
                        {'role': 'user', 'content': user_question},
                    ],
                    format=(ExpectedMultipleAnswer.model_json_schema() if question['multiple_choice'] else ExpectedSingleAnswer.model_json_schema()),
                    options={"temperature": 0.3}
                )
                
                try:
                    response_dict = json.loads(response['message']['content'])
                    if answer_key not in response_dict:
                        continue
                    
                    gemma_answers = response_dict[answer_key]
                    if isinstance(gemma_answers, int):
                        gemma_answers = {gemma_answers}
                    else:
                        gemma_answers = set(gemma_answers)
                    
                    if len(gemma_answers) == 0:
                        continue
                    
                    # Contar acertos
                    q_result.score += len(real_answers & gemma_answers)
                    q_result.total_processed += 1
                    
                    if not question['multiple_choice']:
                        # Escolha única
                        real_ans = list(real_answers)[0]
                        pred_ans = list(gemma_answers)[0]
                        q_result.real_answers_list.append(real_ans)
                        q_result.predicted_answers_list.append(pred_ans)
                    else:
                        # Múltipla escolha
                        for real_alt in real_answers:
                            for pred_alt in gemma_answers:
                                q_result.real_pred_pairs.append((real_alt, pred_alt))
                        
                        for alt_code in question['possible_answers'].values():
                            if alt_code in real_answers and alt_code in gemma_answers:
                                q_result.alternative_scores[alt_code]['tp'] += 1
                            elif alt_code in real_answers and alt_code not in gemma_answers:
                                q_result.alternative_scores[alt_code]['fn'] += 1
                            elif alt_code not in real_answers and alt_code in gemma_answers:
                                q_result.alternative_scores[alt_code]['fp'] += 1
                            else:
                                q_result.alternative_scores[alt_code]['tn'] += 1
                
                except json.JSONDecodeError as e:
                    print(f"  ERRO ao decodificar JSON para {question['code']} (idx {idx}): {e}")
                    continue
                    
            except Exception as e:
                print(f"Erro ao obter resposta para {question['code']} (idx {idx}): {e.__class__.__name__}: {e}")
        
        # Exibir acurácia
        if q_result.total_processed > 0:
            accuracy = q_result.score / (q_result.total_processed * (1 if not question['multiple_choice'] else 3))  # Normaliza pela quantidade máxima de acertos possíveis
        else:
            accuracy = 0.0
        
        print("#" * 50)
        print(f"Fim do Lote para a pergunta {question['code']}")
        print(f"Registros processados: {q_result.total_processed}")
        print(f"Acurácia: {accuracy:.2%}")
        print("#" * 50)
        
        results[question['code']] = q_result
    
    return results


In [18]:
from sklearn.metrics import confusion_matrix

def generate_confusion_matrices(questions: list[dict], results: dict[str, QuestionResults]):
    # Gerar 1 matriz de confusão por pergunta
    for question in questions:

        q_code = question["code"]
        q_results = results[q_code]

        possible_answers = question["possible_answers"]
        labels = list(possible_answers.values())

        real = []
        pred = []

        # Perguntas binárias simples
        if not question["multiple_choice"]:

            real = q_results.real_answers_list
            pred = q_results.predicted_answers_list

        # Perguntas de múltipla escolha
        else:
            # Para a pergunta múltipla escolha:
            # cada par real/predito gera combinações para matriz 6x6
            for real_set, pred_set in q_results.real_pred_pairs:

                # garante iteráveis
                real_values = [real_set]
                pred_values = [pred_set]

                # cria pares alinhados até o menor tamanho
                min_size = min(len(real_values), len(pred_values))

                for i in range(min_size):
                    real.append(real_values[i])
                    pred.append(pred_values[i])

        # Construção da matriz
        cm = confusion_matrix(real, pred, labels=labels)

        # Plot
        plt.figure(figsize=(8, 6))
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=labels,
            yticklabels=labels
        )

        plt.title(f"Matriz de Confusão - {q_code}")
        plt.xlabel("Resposta Predita pelo LLM")
        plt.ylabel("Resposta Real")
        plt.show()


#### Parte 3) Avaliar resultados
Os resultados podem ser avaliados por meio de acurácia e distribuição das respostas.

##### Acurácia

$$
\mathrm{Acurácia} =
\frac{
\sum_{i=1}^{N}
(c_i \cdot w_i)
}{
\sum_{i=1}^{N}
(t_i \cdot w_i)
}
$$

Onde:

- $N$ representa o número total de questões avaliadas;
- $c_i$ representa a quantidade de respostas corretas obtidas na questão $i$;
- $t_i$ representa a quantidade total de respostas esperadas na questão $i$;
- $w_i$ representa o peso associado à questão $i$;
- $w_i = 1$ para questões de escolha única;
- $w_i = 3$ para questões de múltipla escolha.

##### Distirbuição das respostas

Para avaliar a distribuição das respostas, usa-se matrizes de confusão.


In [19]:
batch_size: int = 2

results: dict[str, QuestionResults] = generate_answers_batch(questions, participant_features, df, batch_size)

generate_confusion_matrices(questions, results)

Erro ao obter resposta para P1A (idx 0): ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Erro ao obter resposta para P1A (idx 1): ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
##################################################
Fim do Lote para a pergunta P1A
Registros processados: 0
Acurácia: 0.00%
##################################################
Erro ao obter resposta para P1B (idx 0): ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Erro ao obter resposta para P1B (idx 1): ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
##################################################
Fim do Lote para a pergunta P1B
Registros processados: 0
A

ValueError: Found empty input array (e.g., `y_true` or `y_pred`) while a minimum of 1 sample is required.

# TODO!

In [ ]:
batch_size: int = 50

results_50: dict[str, QuestionResults] = generate_answers_batch(questions, participant_features, df, batch_size)

generate_confusion_matrices(questions, results_50)

In [ ]:
batch_size: int = 100

results_100: dict[str, QuestionResults] = generate_answers_batch(questions, participant_features, df, batch_size)

generate_confusion_matrices(questions, results_100)

In [ ]:
batch_size: int = 200

results_200: dict[str, QuestionResults] = generate_answers_batch(questions, participant_features, df, batch_size)

generate_confusion_matrices(questions, results_200)

Referência:

MIRANDA, F.; BALBI, P. P. Simulating public opinion: Comparing distributional and
individual-level predictions from llms and random forests. *Entropy*, v. 27, n. 9, 2025. ISSN
1099-4300. Disponível em: <https://www.mdpi.com/1099-4300/27/9/923>